# Questão 7 - Previsão de demanda

### Cenário

O Sr. Almir está furioso. No último verão, o estoque de "Coletes Salva-Vidas' acabou em 10 dias, e a
empresa perdeu milhares de reais em vendas. Por outro lado, compraram 'Ancoras" demais e elas
estão enferrujando no galpão. 

Gabriel Santos, o Tech Lead, disse que nao da mais para confiar no "feeling'. Ele quer um modelo preditivo que diga exatamente quantas unidades venderemos no
proximo mes para ajustar as compras com fornecedores.

### Premissas obrigatórias:

- O periodo de treino deve incluir dados até 31/12/2023.
- O periodo de teste deve ser todo o mes de Janeiro de 2024.
- A previsão deve ser feita em base diaria.
- Não é permitido utilizar dados futuros no treino (data leakage).
- Considere apenas o produto: "Motor de Popa Yamaha Evo Dash 155HP"

### Tarefa:

1. Utilize o dataset vendas_2023_2024.csv
2. Construa um modelo baseline simples, utilizando: Média móvel dos últimos 7 dias de vendas (considerando apenas dados anteriores à data prevista).
3. Gere a previsão diária de vendas para Janeiro de 2024.
4. Compare as previsões com os valores reais do período de teste utilizando a métrica: `MAE - Mean Absolute Error`
5. Responda objetivamente:
    * O baseline é adequado para esse produto?
    * Cite uma limitação desse método.

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# Carregar dados
vendas = pd.read_csv('../datasets/vendas_2023_2024.csv')

# Filtrar pelo produto específico (id_product = 54)
produto_id = 54
produto_nome = "Motor de Popa Yamaha Evo Dash 155HP"
vendas_produto = vendas[vendas['id_product'] == produto_id].copy()

# Padronizar coluna de data
vendas_produto['sale_date'] = vendas_produto['sale_date'].str.replace('/', '-')
vendas_produto['date'] = pd.to_datetime(vendas_produto['sale_date'], format='mixed', dayfirst=True)

# Agrupar por dia e somar quantidade
vendas_diarias = vendas_produto.groupby('date')['qtd'].sum().reset_index()

# Definir período de treino e teste
treino = vendas_diarias[vendas_diarias['date'] <= '2023-12-31']
teste = vendas_diarias[(vendas_diarias['date'] >= '2024-01-01') & (vendas_diarias['date'] <= '2024-01-31')]

print(f"Produto: {produto_nome} (ID: {produto_id})")
print(f"Dados de treino: {len(treino)} dias (até 31/12/2023)")
print(f"Dados de teste: {len(teste)} dias (janeiro 2024)")
print(f"Total vendas no período: {vendas_produto['qtd'].sum()} unidades")

Produto: Motor de Popa Yamaha Evo Dash 155HP (ID: 54)
Dados de treino: 28 dias (até 31/12/2023)
Dados de teste: 3 dias (janeiro 2024)
Total vendas no período: 556 unidades


In [2]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -------------------------------------- - 7.9/8.1 MB 51.0 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 32.7 MB/s  0:00:00
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   ----------- ---------------------------- 10.7/37.3 MB 52.9 MB/s eta 0:00:01
   ----------------- ---------------------- 16.0/37.3 MB 46.6 MB/s eta 0:00:01
   ----------------------- ---------------- 22.3/37.3 MB 36.1 MB/s eta 0:00:01
   ---------------------------------- ----- 32.5/37.3 MB 40.0 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.3 MB 40.9 MB/s eta 0:00:01
   ---------------------------------------- 37.3/37.3 MB 30.2 MB/s  0:00:01

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- 

In [7]:
# Implementar baseline: média móvel de 7 dias
def previsao_baseline(data_treino, data_teste):
    previsoes = []
    for i, row in data_teste.iterrows():
        data_atual = row['date']
        # Pegar os últimos 7 dias anteriores à data atual
        ultimos_7_dias = data_treino[data_treino['date'] < data_atual].tail(7)
        if len(ultimos_7_dias) > 0:
            media = ultimos_7_dias['qtd'].mean()
        else:
            media = 0  # Caso não haja dados suficientes
        previsoes.append(media)
    return previsoes

# Gerar previsões para janeiro 2024
previsoes = previsao_baseline(treino, teste)

# Adicionar previsões ao dataframe de teste
teste = teste.copy()
teste['previsao'] = previsoes

print("Previsões geradas:")
print(teste[['date', 'qtd', 'previsao']].head(10))

Previsões geradas:
         date  qtd   previsao
28 2024-01-05   10  11.142857
29 2024-01-21   11  11.142857
30 2024-01-22    6  11.142857


In [8]:
# Calcular MAE
mae = mean_absolute_error(teste['qtd'], teste['previsao'])
print(f"MAE (Mean Absolute Error): {mae:.2f}")

# Estatísticas adicionais
print(f"\nVendas reais médias em janeiro: {teste['qtd'].mean():.2f}")
print(f"Previsões médias: {teste['previsao'].mean():.2f}")
print(f"Total vendas reais janeiro: {teste['qtd'].sum()}")
print(f"Total previsões janeiro: {teste['previsao'].sum():.0f}")

MAE (Mean Absolute Error): 2.14

Vendas reais médias em janeiro: 9.00
Previsões médias: 11.14
Total vendas reais janeiro: 27
Total previsões janeiro: 33


## Respostas às perguntas

1. **O baseline é adequado para esse produto?**  
   Não, o MAE de 2.14 indica um erro médio significativo, especialmente considerando que as vendas diárias são baixas (média de 9 unidades/dia). O método não captura padrões sazonais ou tendências, resultando em previsões imprecisas para um produto com demanda variável.

2. **Cite uma limitação desse método.**  
   A média móvel simples não considera fatores externos como sazonalidade, feriados, condições climáticas ou promoções, que podem influenciar fortemente as vendas de produtos náuticos. Além disso, é reativo e não prevê mudanças de tendência.